In [ ]:
import zipfile
import re
from lxml import etree
from tqdm import tqdm
import pickle
from transformers import AutoTokenizer
import csv
import os
import pandas as pd
import random
import sys
import csv

TSV_FILE = "dta_corpus.tsv"

In [ ]:
def extract_year_from_text(root):
    path = ".//*[local-name()='sourceDesc']//*[local-name()='date' and @type='publication']"
    date_elements = root.xpath(path)
    
    if date_elements and date_elements[0].text:
        match = re.search(r'\d{4}', date_elements[0].text)
        if match:
            return match.group()
            
    return "0000"

def process_zip_directly(zip_path, output_tsv):
    with zipfile.ZipFile(zip_path, 'r') as z:
        file_list = [f for f in z.namelist() if f.endswith(('.tcf', '.xml')) and '/full/' in f]
        
        print(f"Found {len(file_list)} files in the 'full' directory.")
        
        with open(output_tsv, "w", encoding="utf-8") as out:
            for file_name in tqdm(file_list, desc="Processing TCF (Full)"):
                with z.open(file_name) as f:
                    try:
                        tree = etree.parse(f)
                        root = tree.getroot()
                        
                        year = extract_year_from_text(root)

                        tokens = {t.get("ID"): (t.text or "") for t in root.xpath(".//*[local-name()='token']")}
                        lemmas = {l.get("tokenIDs"): (l.text or "") for l in root.xpath(".//*[local-name()='lemma']")}
                        pos_tags = {p.get("tokenIDs"): (p.text or "") for p in root.xpath(".//*[local-name()='tag']")}

                        if not tokens:
                            continue

                        out.write(f"<sod>\t{file_name}\t{year}\n")

                        for sent in root.xpath(".//*[local-name()='sentence']"):
                            t_ids = sent.get("tokenIDs", "").split()
                            for tid in t_ids:
                                word = tokens.get(tid, "")
                                lemma = lemmas.get(tid, word) 
                                pos = pos_tags.get(tid, "UNK")
                                out.write(f"{word}\t{lemma}\t{pos}\n")
                            
                            out.write("<eos>\t<eos>\t<eos>\n")
                            
                    except Exception as e:
                        print(f"Skipping {file_name} due to error: {e}")


In [104]:
if __name__ == "__main__":
    ZIP_PATH = "dta_kernkorpus_2026-02-10_tcf.zip"
    OUTPUT_TSV = "dta_corpus.tsv"
    process_zip_directly(ZIP_PATH, OUTPUT_TSV)

Found 1473 files in the 'full' directory.


Processing TCF (Full):  55%|█████▍    | 810/1473 [31:16<26:47,  2.42s/it]  

Skipping dta_kernkorpus_2026-02-10/full/arnold_ketzerhistorie02_1700.tcf.xml due to error: Error in xpath expression


Processing TCF (Full): 100%|██████████| 1473/1473 [55:51<00:00,  2.28s/it] 


In [105]:
with open("dta_corpus.tsv", encoding="utf-8") as f:
    for _ in range(20):
        print(f.readline())

<sod>	dta_kernkorpus_2026-02-10/full/meissner_krimi_1796.tcf.xml	1796

Kriminal	Kriminal	NN

GESCHICHTEN	Geschichte	NN

von	von	APPR

A	A	NE

G	G	NE

Meißner	Meißner	ADJA

.	.	$.

Wien	Wien	NE

1796	1796	CARD

.	.	$.

<eos>	<eos>	<eos>

I.	i.	ADJA

Mord	Mord	NN

an	an	APPR

ſeiner	seine	PPOSAT

Frau	Frau	NN

,	,	$,

um	um	KOUI

ihre	ihr	PPOSAT



In [ ]:
OUTPUT_PKL = "extracted_targets.pkl"
MODEL_NAME = "dbmdz/bert-base-german-cased"

TARGET_COMPOUNDS = {
    "ruhestand", 
}

class Sentence:
    def __init__(self, tokens, lemmas, tags, year):
        self.tokens = tokens
        self.lemmas = lemmas
        self.tags = tags
        self.year = year
        
    def __repr__(self):
        return f"[{self.year}] {' '.join(self.tokens)}"

def extract_and_pickle():
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    extracted_data = []
    seen_docs = set()
    
    current_tokens, current_lemmas, current_tags = [], [], []
    current_year = None
    skip_doc = False

    print("Scanning TSV for target compounds...")
    with open(TSV_FILE, 'r', encoding='utf-8') as f:
        for line in f:
            parts = line.strip().split('\t')
            if len(parts) != 3:
                continue
            
            tok, lem, tag = parts
            
            if tok == "<sod>":
                doc_id = lem
                if doc_id in seen_docs:
                    skip_doc = True
                else:
                    skip_doc = False
                    seen_docs.add(doc_id)
                    try:
                        current_year = int(tag)
                    except ValueError:
                        skip_doc = True
                        
            elif tok == "<eos>":
                if not skip_doc and current_lemmas:
                    sentence_lemmas_lower = [l.lower() for l in current_lemmas]
                    
                    if any(target in sentence_lemmas_lower for target in TARGET_COMPOUNDS):
                        
                        sent_obj = Sentence(
                            list(current_tokens), 
                            list(current_lemmas), 
                            list(current_tags), 
                            current_year
                        )
                        
                        current_pos = 1
                        
                        for i, lemma in enumerate(current_lemmas):
                            sub_tokens = tokenizer(lemma, add_special_tokens=False)["input_ids"]
                            if not sub_tokens:
                                sub_tokens = [tokenizer.unk_token_id]
                                
                            start_idx = current_pos
                            end_idx = current_pos + len(sub_tokens)
                            
                            if lemma.lower() in TARGET_COMPOUNDS:
                                extracted_data.append({
                                    "sent": sent_obj,
                                    "lemma-span": (start_idx, end_idx),
                                    "target_lemma": lemma.lower()
                                })
                                
                            current_pos = end_idx

                current_tokens.clear()
                current_lemmas.clear()
                current_tags.clear()
                
            elif not skip_doc:
                current_tokens.append(tok)
                current_lemmas.append(lem)
                current_tags.append(tag)

    print(f"Found {len(extracted_data)} target occurrences. Pickling...")
    
    with open(OUTPUT_PKL, "wb") as out_file:
        pickle.dump(extracted_data, out_file, protocol=pickle.HIGHEST_PROTOCOL)
        
    print(f"Done! Saved to {OUTPUT_PKL}")

if __name__ == "__main__":
    extract_and_pickle()

Scanning TSV for target compounds...
Found 933 target occurrences. Pickling...
Done! Saved to extracted_targets.pkl


In [ ]:
class Sentence:
    def __init__(self, tokens, lemmas, tags, year):
        self.tokens = tokens
        self.lemmas = lemmas
        self.tags = tags
        self.year = year
        
    def __repr__(self):
        return f"[{self.year}] {' '.join(self.tokens)}"

def inspect_data(filepath="german_target_pickles/bergwerk.pkl"):
    print(f"Opening {filepath}...\n")
    
    with open(filepath, "rb") as f:
        data = pickle.load(f)
        
    print(f"Data Type: {type(data)}")
    print(f"Total target occurrences extracted: {len(data)}\n")
    
    if len(data) > 0:
        for i in range(min(5, len(data))): 
            item = data[i]
            sent_obj = item["sent"]
            
            print(f"--- ENTRY {i + 1} ---")
            print(f"Target Lemma: '{item['target_lemma']}'")
            print(f"Lemma Span:   {item['lemma-span']}")
            print(f"Year:         {sent_obj.year}")
            print(f"ID:         {sent_obj.sentence_id}")
            
            print(f"Raw Text:     {sent_obj}")
            
            print(f"Tokens List:  {sent_obj.tokens[:10]}...") 
            print(f"Lemmas List:  {sent_obj.lemmas}...")
            print(f"Tags List:    {sent_obj.tags[:10]}...\n")

if __name__ == "__main__":
    inspect_data()

Opening german_target_pickles/bergwerk.pkl...

Data Type: <class 'list'>
Total target occurrences extracted: 2248

--- ENTRY 1 ---
Target Lemma: 'bergwerk'
Lemma Span:   (12, 14)
Year:         1890
ID:         14433
Raw Text:     [1890] Schon 3000 v. Chr. erkämpften sie sich die Bergwerke am Sinai , und der dort sich entwickelnde Bergbau fand eine ungemeine Unterstützung in der Neigung des Arabers zur Handelsthätigkeit .
Tokens List:  ['Schon', '3000', 'v.', 'Chr.', 'erkämpften', 'sie', 'sich', 'die', 'Bergwerke', 'am']...
Lemmas List:  ['schon', '3000', 'v.', 'Chr.', 'erkämpft', 'sie', 'sich', 'd', 'Bergwerk', 'am', 'Sinai', ',', 'und', 'd', 'dort', 'sich', 'entwickelnd', 'Bergbau', 'finden', 'eine', 'ungemein', 'Unterstützung', 'in', 'd', 'Neigung', 'd', 'Araber', 'zur', 'Handelstätigkeit', '.']...
Tags List:    ['ADV', 'CARD', 'APPRART', 'NN', 'ADJA', 'PPER', 'PRF', 'ART', 'NN', 'APPRART']...

--- ENTRY 2 ---
Target Lemma: 'bergwerk'
Lemma Span:   (1, 3)
Year:         1868
ID:      

In [ ]:
class SafeUnpickler(pickle.Unpickler):
    def find_class(self, module, name):
        if module == 'data':
            if not hasattr(sys.modules[__name__], name):
                setattr(sys.modules[__name__], name, type(name, (object,), {}))
            return getattr(sys.modules[__name__], name)
        return super().find_class(module, name)

def generate_german_annotation_samples(directory_path="target_pickles", output_file="german_annotation_tasks.csv", seed=42):
    random.seed(seed)
    
    all_pairs = []
    
    EARLY_RANGE = (1700, 1759)
    LATE_RANGE = (1870, 1909)
    
    def get_obj(item):
        return item.get('sent') if isinstance(item, dict) else getattr(item, 'sent', item)

    def get_year(item):
        return getattr(get_obj(item), 'year', None)

    def get_id(item):
        return getattr(get_obj(item), 'sentence_id', 'N/A')

    def get_text_content(item):
        obj = get_obj(item)
        if hasattr(obj, 'tokens'):
            return " ".join(obj.tokens)
        return str(obj)

    if not os.path.exists(directory_path):
        print(f"Error: Directory '{directory_path}' not found.")
        return

    for filename in os.listdir(directory_path):
        if filename.endswith(".pkl"):
            target_name = filename.replace(".pkl", "").strip()
            file_path = os.path.join(directory_path, filename)
            
            try:
                with open(file_path, 'rb') as f:
                    instances = SafeUnpickler(f).load()
            except Exception as e:
                print(f"Failed to load {filename}: {e}")
                continue
            
            early_instances = [i for i in instances if get_year(i) is not None and EARLY_RANGE[0] <= get_year(i) <= EARLY_RANGE[1]]
            late_instances = [i for i in instances if get_year(i) is not None and LATE_RANGE[0] <= get_year(i) <= LATE_RANGE[1]]
            
            if len(early_instances) < 2 or len(late_instances) < 2:
                print(f"Skipping '{target_name}': Insufficient data (Early: {len(early_instances)}, Late: {len(late_instances)})")
                continue

            target_pairs = []
            
            for _ in range(7):
                s1, s2 = random.sample(early_instances, 2)
                target_pairs.append([
                    target_name, get_id(s1), get_text_content(s1), get_id(s2), get_text_content(s2), "Early-Early"
                ])

            for _ in range(7):
                s1, s2 = random.sample(late_instances, 2)
                target_pairs.append([
                    target_name, get_id(s1), get_text_content(s1), get_id(s2), get_text_content(s2), "Late-Late"
                ])

            for _ in range(7):
                s1 = random.choice(early_instances)
                s2 = random.choice(late_instances)
                target_pairs.append([
                    target_name, get_id(s1), get_text_content(s1), get_id(s2), get_text_content(s2), "Cross-Era"
                ])
            
            all_pairs.extend(target_pairs)

    random.shuffle(all_pairs)
    
    if all_pairs:
        df = pd.DataFrame(all_pairs, columns=[
            'target_word', 'sentence_1_id', 'sentence_1', 'sentence_2_id', 'sentence_2', 'era_group'
        ])
        df.to_csv(output_file, index=False, quoting=csv.QUOTE_ALL)
        print(f"\nSuccess! Generated {len(df)} sentence pairs and saved to '{output_file}'.")
    else:
        print("\nNo pairs were generated. Check your data structure or era filters.")

if __name__ == "__main__":
    generate_german_annotation_samples('german_target_pickles')


Success! Generated 903 sentence pairs and saved to 'german_annotation_tasks.csv'.


In [ ]:
TARGET_COMPOUNDS = {
    "ruhestand", "rechtsstreit", "uhrwerk", "triebwerk", "murmeltier", 
    "eisenwerk", "trauerspiel", "stückwerk", "streitsache", "zeughaus", 
    "gesichtszug", "feuerwerk", "kartenspiel", "wortspiel", "schauspiel", 
    "hausstand", "grundfläche", "meerwasser", "eifersucht", "sündenfall", 
    "sonnenuhr", "tagewerk", "mauerwerk", "sonnenstrahl", "heerführer", 
    "sonnenlicht", "bergwerk", "ziegenbock", "sonnenschein", "kreuzzug", 
    "brunnenwasser", "seewasser", "zitronensaft", "stockwerk", "kinderspiel", 
    "wunderwerk", "bildhauer", "leinöl", "rehbock", "windspiel", 
    "sonnenblume", "salzwasser", "feldzug"
}

def count_target_sentences():
    counts = {target: 0 for target in TARGET_COMPOUNDS}
    current_lemmas = []
    total_sentences_scanned = 0
    
    print(f"Scanning {TSV_FILE} to count target occurrences. This will be quick...")
    
    with open(TSV_FILE, 'r', encoding='utf-8') as f:
        for line in f:
            parts = line.strip().split('\t')
            if len(parts) != 3:
                continue
            
            tok, lem, tag = parts
            
            if tok == "<sod>":
                continue
                
            elif tok == "<eos>":
                if current_lemmas:
                    total_sentences_scanned += 1
                    
                    sentence_lemmas_lower = set(l.lower() for l in current_lemmas)
                    
                    found_targets = TARGET_COMPOUNDS.intersection(sentence_lemmas_lower)
                    
                    for target in found_targets:
                        counts[target] += 1
                        
                current_lemmas.clear()
                
            else:
                current_lemmas.append(lem)

    sorted_counts = sorted(counts.items(), key=lambda x: x[1], reverse=True)
    
    print(f"\nFinished scanning {total_sentences_scanned:,} sentences.")
    print("-" * 40)
    print(f"{'Target Compound':<20} | {'Sentence Count'}")
    print("-" * 40)
    
    for target, count in sorted_counts:
        print(f"{target:<20} | {count:,}")
        
    total_extracted = sum(count for count in counts.values())
    print("-" * 40)
    print(f"Total target instances found: {total_extracted:,}")

if __name__ == "__main__":
    count_target_sentences()

Scanning dta_corpus.tsv to count target occurrences. This will be quick...

Finished scanning 6,136,396 sentences.
----------------------------------------
Target Compound      | Sentence Count
----------------------------------------
schauspiel           | 3,188
bergwerk             | 2,118
feldzug              | 1,970
sonnenschein         | 1,801
eifersucht           | 1,624
sonnenstrahl         | 1,265
bildhauer            | 1,162
stockwerk            | 1,137
wunderwerk           | 1,079
trauerspiel          | 1,075
eisenwerk            | 1,037
ruhestand            | 928
mauerwerk            | 832
sonnenlicht          | 724
rechtsstreit         | 694
salzwasser           | 652
feuerwerk            | 605
grundfläche          | 593
gesichtszug          | 506
heerführer           | 468
zeughaus             | 419
kreuzzug             | 380
uhrwerk              | 374
brunnenwasser        | 284
tagewerk             | 280
meerwasser           | 262
kinderspiel          | 241
hausstand     